# **금융경제학**

- 금융경제학 (박기영 저, 시그마프레스) 교재에 사용된 데이터/모형/그래프 관련 작업을 수행하는 python notebook 파일임: https://github.com/FinancialEconomicsPython
- python 코드는 구글 코랩에서 사용하는 것을 기준으로 작성되었음.
- 데이터 파일이 필요한 경우 위치: https://drive.google.com/drive/folders/1sArqUZKnxWtkNtHe31iD1w-2xCVEhTj0?usp=share_link
- date: 2025/3/22, updated: 2026/2/7

# 사전준비

## 실행 방법

**아래 셀 하나만 실행**하면 준비가 끝납니다 — 리포 경로 자동 감지, 라이브러리 설치, 그래프 스타일, 한글 폰트, NBER 경기침체 데이터 로딩까지 한 번에 처리합니다.

준비할 것은 **한국은행 ECOS API 키** 하나뿐입니다.

- 키가 아직 없다면: https://ecos.bok.or.kr/api/#/ 에서 회원가입 후 **[인증키 신청]**을 하면 됩니다(무료).

발급받은 키를 노트북에 알려주는 방법은 두 가지입니다. 하나만 하면 됩니다.

**방법 1 (권장) — Colab에 한 번만 등록해 두기**

1. Colab 화면 **왼쪽 세로 메뉴의 열쇠 모양 아이콘(🔑)**을 클릭하면 '보안 비밀(Secrets)' 창이 열립니다.
2. **[+ 새 보안 비밀 추가]**를 누르고 **이름**에 `ECOS_API_KEY`, **값**에 발급받은 키를 붙여넣습니다.
3. 그 행 왼쪽의 **'노트북 액세스' 스위치를 켭니다.**

한 번 등록해 두면 이 교재의 **모든 장에서 자동으로 사용**되고, 키가 노트북 파일에 저장되지 않아 노트북을 공유해도 키가 노출되지 않습니다.

**방법 2 — 아래 셀에 직접 입력**

- 아래 셀 마지막 줄의 `ecos_key="YOUR_ECOS_API_KEY_HERE"`에서 따옴표 안을 본인 키로 바꿉니다.
- 이 방법은 노트북 파일에 키가 남으므로, 노트북을 다른 사람과 공유할 때 키도 함께 노출된다는 점만 주의하세요.

참고
- 리포를 git clone 했거나(README 방법 A) Colab에서 실행하면 경로(BASE/UTILS/FIGS/DATA)는 **자동으로 잡히므로 직접 지정할 필요가 없습니다.**
- 준비 로직의 상세 구현은 `utils/bootstrap.py`에 있습니다.


In [ ]:
# ============================================================
# 부트스트랩 — 모든 장 공통 (상세 구현: utils/bootstrap.py)
# ============================================================
import os, sys

# 리포 루트 자동 감지: (1) 이미 clone됨 → (2) Google Drive(저자) → (3) 자동 clone → (4) 로컬
if os.path.isdir("/content") and not os.path.exists("/content/resources"):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass

_DRIVE = "/content/drive/MyDrive/Colab Notebooks/book_FinancialEconomics"
if os.path.exists("/content/resources"):
    ROOT = "/content/resources"          # 독자: git clone 방식(README 방법 A)
elif os.path.isdir(_DRIVE):
    ROOT = _DRIVE                        # 저자: Google Drive 방식
elif os.path.isdir("/content"):
    os.system("git clone -q https://github.com/FinancialEconomicsPython/resources.git /content/resources")
    ROOT = "/content/resources"          # 독자: 자동 clone
else:
    ROOT = os.getcwd()                   # 로컬 실행

sys.path.insert(0, os.path.join(ROOT, "utils"))
from bootstrap import init

# ECOS API 키: Colab Secrets(🔑)에 'ECOS_API_KEY' 저장을 권장. 없으면 아래에 직접 입력.
env = init(globals(), root=ROOT, ecos_key="YOUR_ECOS_API_KEY_HERE")


# Main

## 상대적 위험기피계수와 확실성 등가

In [ ]:
import numpy as np
import pandas as pd

def certainty_equivalent_crra(x1, x2, p=0.5, gamma=1.0):
    """
    CE X for lottery: p*x1 + (1-p)*x2 under CRRA utility.

    If gamma == 1: u(c)=ln(c)
    Else: u(c)=c^(1-gamma)/(1-gamma)
    """
    x1 = float(x1)
    x2 = float(x2)

    if gamma == 1:
        # p*ln(x1) + (1-p)*ln(x2) = ln(X)  ->  X = exp( p ln x1 + (1-p) ln x2 )
        return np.exp(p*np.log(x1) + (1-p)*np.log(x2))
    else:
        # p*u(x1) + (1-p)*u(x2) = u(X)
        EU = p*(x1**(1-gamma))/(1-gamma) + (1-p)*(x2**(1-gamma))/(1-gamma)
        # invert u: X^(1-gamma)/(1-gamma) = EU  ->  X^(1-gamma) = (1-gamma)*EU
        return ((1-gamma)*EU)**(1/(1-gamma))

# -----------------------------
# 입력: 5천만원, 1억원 (단위: "만원")
# -----------------------------
x_low  = 5000    # 5천만원
x_high = 10000   # 1억원
p = 0.5

gammas = [1, 3, 5, 10, 20, 30]

rows = []
for g in gammas:
    X = certainty_equivalent_crra(x_low, x_high, p=p, gamma=g)
    rows.append([g, X])

df_table = pd.DataFrame(rows, columns=['상대적 위험기피계수(γ)', 'X'])

# 표 표시용 포맷: 반올림 후 '만원' 붙이기
df_table['X'] = df_table['X'].round(0).astype(int).map(lambda v: f"{v:,}만원")

df_table

,상대적 위험기피계수(γ),X
0,1,"7,071만원"
1,3,"6,325만원"
2,5,"5,857만원"
3,10,"5,399만원"
4,20,"5,186만원"
5,30,"5,121만원"
